In [1]:
# 패키지 불러오기

import os  # 파일 존재 여부 확인 및 경로 관련 처리를 위한 표준 라이브러리
import re  # 정규식을 이용한 텍스트 정제용 라이브러리
import urllib.request  # 학습 데이터를 인터넷에서 다운로드하기 위한 라이브러리
import tarfile  # tar.gz 형식의 압축 파일을 풀기 위한 라이브러리
import torch  # PyTorch 메인 패키지
import torch.nn as nn  # 신경망 레이어를 정의하기 위한 모듈
import torch.optim as optim  # 옵티마이저(Adam 등)를 사용하기 위한 모듈
from torch.utils.data import DataLoader, Dataset  # 데이터셋/배치 처리를 위한 유틸리티
from tqdm import tqdm  # 학습/전처리 진행 상황을 보기 위한 진행 바
import numpy as np  # (현재는 사용하지 않지만) 수치 연산용 패키지
import matplotlib.pyplot as plt  # (현재는 사용하지 않지만) 시각화를 위한 패키지
import matplotlib.ticker as ticker  # (현재는 사용하지 않지만) 그래프 축 설정용 유틸리티
from pecab import PeCab  # 한국어 형태소 분석기 (토큰화에 사용)

In [2]:
# -------------------------------------------------------------------------------------------------------
# Step 1 & 2: Data Loading and Cleaning
# -------------------------------------------------------------------------------------------------------

DATA_URL = "https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz"  # 코퍼스 다운로드 URL
DATA_FILENAME = "korean-english-park.train.tar.gz"  # 압축 파일 이름
KO_PATH = "korean-english-park.train.ko"  # 압축 해제 후 한국어 파일 이름
EN_PATH = "korean-english-park.train.en"  # 압축 해제 후 영어 파일 이름
PREPROCESSED_PATH = "preprocessed_corpus.pt"  # 전처리(토큰화) 결과를 캐싱할 파일 경로

def download_data(url, filename):
    """주어진 URL에서 tar.gz 파일을 다운로드하는 함수."""
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")

def extract_data(filename):
    """다운로드 받은 tar.gz 파일을 현재 디렉토리로 모두 압축 해제하는 함수."""
    print(f"Extracting {filename}...")
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall()
    print("Extraction complete.")

if not os.path.exists(KO_PATH) or not os.path.exists(EN_PATH):
    # 병렬 코퍼스가 없으면 다운로드 및 압축 해제 수행
    if not os.path.exists(DATA_FILENAME):
        download_data(DATA_URL, DATA_FILENAME)
    extract_data(DATA_FILENAME)
    
def load_data(ko_path, en_path):
    """
    한국어/영어 병렬 코퍼스를 각각 읽어 들인 뒤,
    (한국어 문장, 영어 문장) 쌍을 만들어 중복을 제거해서 반환한다.
    """
    with open(ko_path, "r", encoding="utf-8") as f:
        ko_corpus = f.read().splitlines()
    with open(en_path, "r", encoding="utf-8") as f:
        en_corpus = f.read().splitlines()
        
    # (한국어, 영어) 문장 쌍 단위로 set을 만들어 중복 제거
    cleaned_corpus = list(set(zip(ko_corpus, en_corpus)))
    return cleaned_corpus

def preprocess_sentence(sentence, is_ko=False):
    """
    한 문장을 입력으로 받아, 대소문자 정규화 및 불필요한 문자를 제거해
    모델이 다루기 좋은 형태로 정제하는 함수.

    is_ko=True  이면 한국어용 정규식을,
    is_ko=False 이면 영어용 정규식을 적용한다.
    """
    sentence = sentence.lower().strip()  # 소문자로 변경하고 양쪽 공백 제거
    
    if is_ko:
        # 한국어 정제: 한글, 숫자, 기본적인 문장 부호만 남기기
        sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
        sentence = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣0-9?.!,]+", " ", sentence)
    else:
        # 영어 정제: 알파벳, 숫자, 기본적인 문장 부호만 남기기
        sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
        sentence = re.sub(r"[^a-zA-Z0-9?.!,]+", " ", sentence)
        
    sentence = re.sub(r'[" "]+', " ", sentence)  # 여러 개의 공백을 하나로 합치기
    return sentence.strip()  # 앞뒤 공백 제거 후 반환

def tokenize_corpus(corpus):
    """
    (한국어 문장, 영어 문장) 쌍 리스트를 입력받아
    - 한국어는 PeCab으로 형태소 단위 토큰화
    - 영어는 공백 기준 토큰화 + <start>, <end> 토큰 부착
    - 최대 길이 40 토큰 이하인 문장 쌍만 필터링
    을 수행하고 토큰화된 코퍼스를 반환한다.
    """
    kor_corpus = []
    eng_corpus = []
    
    for ko, en in tqdm(corpus, desc="Preprocessing"):
        ko_pre = preprocess_sentence(ko, is_ko=True)   # 한국어 문장 정제
        en_pre = preprocess_sentence(en, is_ko=False)  # 영어 문장 정제
        
        # 한국어 문장을 형태소 단위로 분리
        ko_tokens = pecab.morphs(ko_pre)
        
        # 영어 문장을 공백 단위로 나누고, 문장 시작/끝 토큰 추가
        en_tokens = ["<start>"] + en_pre.split() + ["<end>"]
        
        # 길이가 40 토큰 이하인 문장 쌍만 사용
        if len(ko_tokens) <= 40 and len(en_tokens) <= 40:
            kor_corpus.append(ko_tokens)
            eng_corpus.append(en_tokens)
            
    return kor_corpus, eng_corpus

print("Loading and cleaning data...")
cleaned_corpus = load_data(KO_PATH, EN_PATH)  # 파일에서 문장 쌍을 읽고 중복 제거

pecab = PeCab()  # 한국어 형태소 분석기 초기화

Loading and cleaning data...


In [3]:
# -----------------------------
# 전처리(토큰화) 결과 캐싱 로직
# -----------------------------
if os.path.exists(PREPROCESSED_PATH):
    # 캐시 파일이 있으면, 긴 형태소 분석 과정을 생략하고 바로 로드
    print("Loading tokenized corpus from cache...")
    cached = torch.load(PREPROCESSED_PATH)
    kor_corpus = cached["kor_corpus"]
    eng_corpus = cached["eng_corpus"]
else:
    # 캐시가 없으면 처음 한 번만 전체 코퍼스를 토큰화하고, 그 결과를 저장
    print("Tokenizing corpus (this may take a while)...")
    kor_corpus, eng_corpus = tokenize_corpus(cleaned_corpus)  # 토큰화된 한국어/영어 코퍼스
    print(f"Saving tokenized corpus to cache: {PREPROCESSED_PATH}")
    torch.save({"kor_corpus": kor_corpus, "eng_corpus": eng_corpus}, PREPROCESSED_PATH)
print(f"Total pairs after filtering: {len(kor_corpus)}")

Loading tokenized corpus from cache...
Total pairs after filtering: 59406


In [4]:
# -------------------------------------------------------------------------------------------------------
# Step 3: Tokenization to Tensors
# -------------------------------------------------------------------------------------------------------

class Tokenizer:
    """
    간단한 단어 기반 토크나이저.
    - word2idx : 단어 → 정수 ID 매핑
    - idx2word : 정수 ID → 단어 매핑
    최빈도 순으로 vocab_size 만큼의 단어를 사전에 등록한다.
    """

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size  # 단어 사전 최대 크기
        # 특수 토큰들: 패딩, 미등록 단어, 문장 시작/끝
        self.word2idx = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
        self.idx2word = {0: "<pad>", 1: "<unk>", 2: "<start>", 3: "<end>"}
        self.counts = {}  # 단어 빈도 수를 저장할 딕셔너리

    def build_vocab(self, corpus):
        """
        말뭉치 전체를 돌면서 단어 빈도를 센 후,
        가장 많이 등장한 단어부터 vocab_size 개수만큼 사전에 추가한다.
        """
        for sentence in corpus:
            for word in sentence:
                self.counts[word] = self.counts.get(word, 0) + 1
        
        # 빈도수 기준 내림차순 정렬 후, 상위 vocab_size 개수만 사용
        sorted_words = sorted(self.counts.items(), key=lambda x: x[1], reverse=True)
        for word, _ in sorted_words[:self.vocab_size - 4]:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, sentence):
        """
        토큰(단어) 리스트를 정수 ID 리스트로 변환한다.
        사전에 없는 단어는 <unk>(1) 토큰 ID로 대체한다.
        """
        return [self.word2idx.get(word, 1) for word in sentence]

    def decode(self, ids):
        """
        정수 ID 리스트를 다시 토큰(단어) 리스트로 변환한다.
        """
        return [self.idx2word.get(id, "<unk>") for id in ids]

# 토크나이저 생성 (요청 사항에 따라 10,000개 이상으로 설정)
VOCAB_SIZE = 12000
kor_tokenizer = Tokenizer(VOCAB_SIZE)  # 한국어용 토크나이저
eng_tokenizer = Tokenizer(VOCAB_SIZE)  # 영어용 토크나이저

kor_tokenizer.build_vocab(kor_corpus)  # 한국어 빈도 기반 단어 사전 생성
eng_tokenizer.build_vocab(eng_corpus)  # 영어 빈도 기반 단어 사전 생성

def sequences_to_tensor(corpus, tokenizer, max_len=40):
    """
    토큰화된 문장 리스트를 고정 길이 텐서로 변환하는 함수.
    - 각 문장은 토크나이저로 정수 ID 시퀀스로 변환
    - 길이가 max_len 보다 짧으면 패딩(0) 추가
    - 길이가 max_len 보다 길면 잘라냄
    """
    tensor = []
    for sentence in corpus:
        ids = tokenizer.encode(sentence)
        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))  # 패딩 토큰(0)으로 길이 맞추기
        else:
            ids = ids[:max_len]
        tensor.append(ids)
    return torch.tensor(tensor)

kor_tensor = sequences_to_tensor(kor_corpus, kor_tokenizer)  # 한국어 입력 텐서
eng_tensor = sequences_to_tensor(eng_corpus, eng_tokenizer)  # 영어 출력(정답) 텐서

print(f"Korean tensor shape: {kor_tensor.shape}")
print(f"English tensor shape: {eng_tensor.shape}")

Korean tensor shape: torch.Size([59406, 40])
English tensor shape: torch.Size([59406, 40])


In [5]:
# -------------------------------------------------------------------------------------------------------
# Step 4: Model Design (Seq2Seq with Attention)
# -------------------------------------------------------------------------------------------------------

class BahdanauAttention(nn.Module):
    """
    Bahdanau(어텐션) 메커니즘 구현.
    디코더의 현재 hidden state와 인코더의 모든 출력 값을 이용해
    각 타임스텝에 대한 가중치(어텐션 분포)를 계산한다.
    """

    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)  # 인코더 출력 변환용 가중치
        self.W2 = nn.Linear(hidden_dim, hidden_dim)  # 디코더 hidden 변환용 가중치
        self.v = nn.Linear(hidden_dim, 1, bias=False)  # energy를 스칼라 점수로 변환

    def forward(self, hidden, encoder_outputs):
        """
        hidden: (batch_size, hidden_dim)       - 디코더의 현재 hidden state
        encoder_outputs: (src_len, batch_size, hidden_dim) - 인코더의 전체 출력
        """
        src_len = encoder_outputs.shape[0]
        # hidden을 시퀀스 길이만큼 반복해서 encoder_outputs와 shape을 맞춰줌
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim)

        # energy 계산 후, v를 통과시켜 각 타임스텝별 스칼라 점수 산출
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)
        return torch.softmax(attention, dim=1)  # 각 타임스텝에 대한 어텐션 가중치


class Encoder(nn.Module):
    """
    단순 GRU 기반 인코더.
    - 입력: (src_len, batch_size)
    - 출력: 모든 타임스텝의 hidden state와 마지막 hidden state
    """

    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)  # 단어 ID → 임베딩 벡터
        self.rnn = nn.GRU(emb_dim, hidden_dim)  # GRU 인코더

    def forward(self, src):
        embedded = self.embedding(src)  # (src_len, batch_size, emb_dim)
        outputs, hidden = self.rnn(embedded)  # outputs: 모든 타임스텝, hidden: 마지막 state
        return outputs, hidden


class Decoder(nn.Module):
    """
    어텐션을 사용하는 GRU 기반 디코더.
    - 매 타임스텝마다 인코더 출력 전체에 어텐션을 적용해 컨텍스트 벡터를 만들고,
      이를 현재 출력과 합쳐 다음 토큰의 분포를 예측한다.
    """

    def __init__(self, vocab_size, emb_dim, hidden_dim, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, emb_dim)  # 디코더 입력 임베딩
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        # 디코더 출력(hidden_dim) + 컨텍스트 벡터(hidden_dim)를 이어 붙여 단어 분포 예측
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, vocab_size)

    def forward(self, input, hidden, encoder_outputs):
        """
        input : (batch_size,)        - 현재 시점에 넣을 단어 ID
        hidden: (1, batch_size, hidden_dim) - 이전 시점의 hidden state
        encoder_outputs: (src_len, batch_size, hidden_dim) - 인코더 전체 출력
        """
        input = input.unsqueeze(0)  # (1, batch_size)
        embedded = self.embedding(input)  # (1, batch_size, emb_dim)

        # 현재 디코더 hidden과 인코더 출력을 이용해 어텐션 가중치 계산
        a = self.attention(hidden[-1], encoder_outputs).unsqueeze(1)  # (batch_size, 1, src_len)
        enc_out = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim)
        # 배치 행렬 곱으로 컨텍스트 벡터 계산 → (batch_size, 1, hidden_dim)
        context = torch.bmm(a, enc_out).permute(1, 0, 2)  # (1, batch_size, hidden_dim)

        # RNN 한 스텝 진행
        output, hidden = self.rnn(embedded, hidden)

        # 차원 축소 (시퀀스 길이 1 제거)
        output = output.squeeze(0)   # (batch_size, hidden_dim)
        context = context.squeeze(0)  # (batch_size, hidden_dim)

        # 디코더 출력과 컨텍스트를 이어붙인 후 단어 분포 예측
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, vocab_size)

        # prediction: 현재 시점의 단어 분포, hidden: 다음 시점에 넘겨줄 hidden, a: 어텐션 분포
        return prediction, hidden, a.squeeze(1)


class Seq2SeqAttention(nn.Module):
    """
    Encoder + Attention Decoder를 하나로 묶은 Seq2Seq 모델.
    - 학습 시(trg 인자가 있을 때)는 Teacher Forcing 형태로 전체 정답 시퀀스를 이용
    - 추론 시(trg=None)은 <start> 토큰부터 시작해 <end> 토큰이 나올 때까지 생성
    """

    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=40):
        batch_size = src.shape[1]  # 배치 크기 (두 번째 차원)
        outputs = []   # 각 타임스텝의 디코더 출력을 모을 리스트
        attentions = []  # 추론 시 어텐션 가중치를 저장 (시각화 등에 활용 가능)

        # 1) 인코더를 한 번 통과해 전체 시퀀스의 hidden state를 얻음
        enc_output, hidden = self.encoder(src)

        if trg is not None:
            # --------- 학습 모드(Teacher Forcing) ---------
            for t in range(trg.shape[0]):
                input = trg[t]  # 정답 문장의 t번째 단어를 현재 입력으로 사용
                output, hidden, a = self.decoder(input, hidden, enc_output)
                outputs.append(output.unsqueeze(0))  # (1, batch_size, vocab_size) 형태로 쌓기
        else:
            # --------- 추론 모드(번역 생성) ---------
            # 시작 토큰(<start> = ID 2)에서 시작
            input = torch.full((batch_size,), 2, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)  # <end> 여부 체크용

            for t in range(max_len):
                output, hidden, a = self.decoder(input, hidden, enc_output)
                outputs.append(output.unsqueeze(0))
                attentions.append(a.unsqueeze(0))

                # 현재 시점에서 가장 확률이 높은 단어를 다음 입력으로 사용
                input = output.argmax(1)
                # <end> 토큰(ID 3)을 만나면 해당 배치 요소는 종료로 표시
                finished |= (input == 3)
                if finished.all():  # 모든 문장이 <end>를 만났다면 조기 종료
                    break

        # outputs: (시퀀스 길이, batch_size, vocab_size)
        return torch.cat(outputs, dim=0), (torch.cat(attentions, dim=0) if attentions else None)

In [9]:
# -------------------------------------------------------------------------------------------------------
# Step 5: Training
# -------------------------------------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 사용 가능 시 cuda, 아니면 cpu
EMB_DIM = 256  # 임베딩 차원 수
HID_DIM = 512  # GRU hidden state 차원 수

# 인코더/디코더/어텐션/전체 모델 초기화
encoder = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM).to(device)
attention = BahdanauAttention(HID_DIM).to(device)
decoder = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM, attention).to(device)
model = Seq2SeqAttention(encoder, decoder, device).to(device)

# Adam 옵티마이저와 CrossEntropyLoss 설정
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 패딩 토큰(0)은 손실 계산에서 무시

# Learning rate 스케줄러 추가
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",   # loss가 감소하면 개선으로 판단
    factor=0.5,   # 개선 없을 때 lr을 절반으로 감소
    patience=1,   # 1 epoch 동안 개선이 없으면 감소
)

# Checkpoint 기능 추가
CHECKPOINT_PATH = "checkpoint.pt"

start_epoch = 0
best_loss = float("inf")

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    start_epoch = ckpt.get("epoch", 0)
    best_loss = ckpt.get("best_loss", float("inf"))
    print(f"Resuming from epoch {start_epoch}, best_loss={best_loss:.4f}")
    print(f"지금까지 최소 {start_epoch} epoch 학습된 모델입니다.")
else:
    print("Checkpoint 파일이 없어 처음부터 학습을 시작합니다.")

class TranslationDataset(Dataset):
    """
    PyTorch Dataset 형태로 한국어/영어 텐서를 묶어주는 클래스.
    - __getitem__ 에서 (src, trg) 쌍을 반환한다.
    """

    def __init__(self, src_tensor, trg_tensor):
        self.src = src_tensor  # 한국어 입력 텐서
        self.trg = trg_tensor  # 영어 정답 텐서

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.trg[idx]


dataset = TranslationDataset(kor_tensor, eng_tensor)  # 전체 병렬 코퍼스를 Dataset으로 감싸기
loader = DataLoader(dataset, batch_size=64, shuffle=True)  # 미니배치 생성용 DataLoader


def translate(sentence, model, kor_tok, eng_tok, device, max_len=40):
    """
    한국어 문장을 입력으로 받아 모델을 이용해 영어 번역 문장을 생성하는 함수.
    1) 입력 문장 정제 및 토큰화
    2) 토큰을 정수 ID로 변환 후 텐서 변환
    3) 모델을 통해 디코더 출력 시퀀스를 얻고,
    4) <end> 토큰이 나오기 전까지의 단어를 디코딩해 문장으로 합친다.
    """
    model.eval()  # 평가 모드로 전환 (Dropout/BatchNorm 비활성화)
    pre = preprocess_sentence(sentence, is_ko=True)  # 한국어 문장 정제
    tokens = pecab.morphs(pre)  # 형태소 단위 토큰화
    ids = kor_tok.encode(tokens)  # 정수 ID 시퀀스로 변환
    src_tensor = torch.tensor(ids).unsqueeze(1).to(device)  # (seq_len, 1) 모양의 텐서

    with torch.no_grad():
        # trg=None 이므로 추론 모드로 동작하며, max_len 까지 번역 생성
        output, _ = model(src_tensor, max_len=max_len)

    # output: (생성길이, batch=1, vocab_size) → 가장 확률 높은 토큰 ID 선택
    pred_ids = output.argmax(2).squeeze(1).cpu().tolist()
    pred_tokens = eng_tok.decode(pred_ids)  # ID 시퀀스를 단어 시퀀스로 변환

    # <end> 토큰이 나오면 그 이전까지만 문장으로 사용
    if "<end>" in pred_tokens:
        pred_tokens = pred_tokens[:pred_tokens.index("<end>")]

    return " ".join(pred_tokens)

Loading checkpoint from checkpoint.pt...
Resuming from epoch 30, best_loss=0.4794
지금까지 최소 30 epoch 학습된 모델입니다.


### < 학습 실행 >

In [10]:
print("\nStarting Training...")

EPOCHS = 35
for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0
    progress = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for src, trg in progress:
        # DataLoader는 (batch, seq_len) 형태로 주기 때문에 (seq_len, batch)로 전치
        src, trg = src.permute(1, 0).to(device), trg.permute(1, 0).to(device)              ################################
        optimizer.zero_grad()

        # trg[:-1, :] : 입력(마지막 토큰 제외), trg[1:, :] : 라벨(첫 토큰 제외, 한 칸 오른쪽으로 시프트된 형태)
        output, _ = model(src, trg[:-1, :])
        # output: (seq_len-1, batch, vocab_size) → (전체 토큰 수, vocab_size)로 펼치기
        output = output.view(-1, VOCAB_SIZE)
        # trg_label: (seq_len-1, batch) → (전체 토큰 수,)로 펼치기
        trg_label = trg[1:, :].reshape(-1)                                 #################################################

        loss = criterion(output, trg_label)  # CrossEntropyLoss 계산
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient 폭주 방지용 클리핑
        optimizer.step()

        epoch_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(loader)
    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

    # learning rate 스케줄러에 현재 epoch의 평균 loss 전달
    scheduler.step(avg_loss)

    # 가장 좋은(loss가 최소) 모델만 저장
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(
            {
                "epoch": epoch + 1,          # 다음에 시작할 epoch
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "best_loss": best_loss,
            },
            CHECKPOINT_PATH,
        )
        print(f"Checkpoint saved at epoch {epoch+1} (best_loss={best_loss:.4f})")

    # 매 epoch 종료 후 간단한 예시 문장을 번역해보며 학습 상황 확인
    print(f"1) 오바마: {translate('오바마는 대통령이다.', model, kor_tokenizer, eng_tokenizer, device)}")
    print(f"2) 시민들: {translate('시민들은 도시 속에 산다.', model, kor_tokenizer, eng_tokenizer, device)}")


Starting Training...


Epoch 31/35: 100%|██████████| 929/929 [03:25<00:00,  4.52it/s, loss=0.623]


Epoch 31 Loss: 0.4738
Checkpoint saved at epoch 31 (best_loss=0.4738)
1) 오바마: obama is on the obama s economic cooperation .
2) 시민들: <unk> took place in the city and cities in <unk> .


Epoch 32/35: 100%|██████████| 929/929 [03:24<00:00,  4.54it/s, loss=0.602]


Epoch 32 Loss: 0.4694
Checkpoint saved at epoch 32 (best_loss=0.4694)
1) 오바마: the obama presidency carries democratic presidential primary , obama s means .
2) 시민들: the black <unk> it gets off the cities , locals and the <unk> .


...

Epoch 33 Loss: 0.4653
Checkpoint saved at epoch 33 (best_loss=0.4653)
1) 오바마: obama is on the obama s economic cooperation .
2) 시민들: the black <unk> it gets off cities and it <unk> .


Epoch 34/35: 100%|██████████| 929/929 [03:24<00:00,  4.54it/s, loss=0.674]


Epoch 34 Loss: 0.4634
Checkpoint saved at epoch 34 (best_loss=0.4634)
1) 오바마: it is obama s economic crisis , with him saying .
2) 시민들: the citizens took place in the city of nice city .


Epoch 35/35: 100%|██████████| 929/929 [03:24<00:00,  4.53it/s, loss=0.584]


Epoch 35 Loss: 0.4598
Checkpoint saved at epoch 35 (best_loss=0.4598)
1) 오바마: obama is the latest in the latest , saying it has close him .
2) 시민들: cities like towns and cities won t get crowded residents there .


In [11]:
# translation test

print("\nFinal Evaluation:")
test_cases = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]
for tc in test_cases:
    print(f"K: {tc} -> E: {translate(tc, model, kor_tokenizer, eng_tokenizer, device)}")


Final Evaluation:
K: 오바마는 대통령이다. -> E: obama is the latest in the latest , saying it has close him .
K: 시민들은 도시 속에 산다. -> E: cities like towns and cities won t get crowded residents there .
K: 커피는 필요 없다. -> E: for coffee , maybe you have is no choice but to do it is necessary .
K: 일곱 명의 사망자가 발생했다. -> E: the deaths of seven people died at the six seven main <unk> injuries .
